## 02 — Silver: validation & enrichment

Applies **business** validity rules and joins reference data to add business context.

**Input:** `sncf_gc.bronze.frequentation`, `.jours_feries`, `.population_communes`

**Output:** `sncf_gc.silver.frequentation_clean`

### Validity rules
Drops rows that can't be used for business aggregation: missing key
(date, gare_id), missing measure (nb_voyageurs), impossible negative
counts, and out-of-domain segment values.

In [0]:
from pyspark.sql import functions as F

VALID_SEGMENTS = {"A", "B", "C"}  # adjust to your real segment values

In [0]:
def validate_frequentation(df):
    """Apply business validity rules to typed Bronze data.

    Drops rows that are not exploitable for analysis: missing key
    (date, gare_id), missing measure (nb_voyageurs), impossible
    negative counts, and out-of-domain segment values.

    Returns:
        DataFrame keeping only rows valid for business aggregation.
    """
    return (
        df
        .dropna(subset=["date", "gare_id", "nb_voyageurs"])
        .filter((F.col("nb_voyageurs") >= 0) & (F.col("nb_non_voyageurs") >= 0))
        .filter(F.col("segment").isin(*VALID_SEGMENTS))
    )

### Enrichment
Adds a public-holiday flag (footfall differs on holidays) and commune
population (enables per-capita normalization in Gold). Left joins keep
every footfall row even without a reference match.

In [0]:
def enrich_frequentation(df, jours_feries, population):
    """Enrich validated footfall with business context.

    Adds a public-holiday flag (footfall behaves differently on holidays)
    and commune population (enables per-capita normalization in Gold).
    Left joins preserve every footfall row even without a reference match.

    Returns:
        Enriched DataFrame [+ est_jour_ferie, population, code_insee].
    """
    holidays = jours_feries.select("date").withColumn("est_jour_ferie", F.lit(True))

    return (
        df
        .join(holidays, on="date", how="left")
        .withColumn("est_jour_ferie", F.coalesce(F.col("est_jour_ferie"), F.lit(False)))
        .join(
            population.select(
                F.col("nom_commune").alias("ville"), "code_insee", "population"
            ),
            on="ville",
            how="left",
        )
    )

In [0]:
df_silver = enrich_frequentation(
    validate_frequentation(spark.table("sncf_gc.bronze.frequentation")),
    spark.table("sncf_gc.bronze.jours_feries"),
    spark.table("sncf_gc.bronze.population_communes"),
)

df_silver.write.format("delta").mode("overwrite").saveAsTable("sncf_gc.silver.frequentation_clean")
print(f"✅ silver.frequentation_clean: {spark.table('sncf_gc.silver.frequentation_clean').count()} rows")

In [0]:
spark.table("sncf_gc.bronze.frequentation")